# OLS vs. GLS one more time

**Ordinary Least Squares (OLS)** assumes that the error terms have constant variance (homoskedasticity) and no correlation across observations. However, in many real-world applications, these assumptions are violated, leading to inefficient and potentially biased standard errors.

**Generalized Least Squares (GLS)** corrects for heteroskedasticity and autocorrelation by transforming the model so that the transformed errors satisfy the OLS assumptions.
In this exercise, you will:

1. Simulate a dataset where OLS assumptions are violated.
2. Fit an OLS model and observe inefficiencies.
3. Apply GLS to correct the estimation.

## Step 1: Simulating Data with Heteroskedastic Errors

We create a simple regression model:

$$
Y_i = \beta_0 + \beta_1 X_i + \epsilon_i, \quad \epsilon_i \sim N(0, \sigma_i^2)
$$

where the error variance depends on $X$:

$$
\sigma_i^2 = \alpha X_i^3
$$


In [ ]:
library(data.table)

# Set seed for reproducibility
set.seed(42)

# Create a data.table with n observations
n <- 1000
dt <- data.table(X = runif(n, 1, 10))

In [ ]:
# Define heteroskedastic errors: variance increases with X
sigma_sq <- 5 * dt$X^3 
dt[, epsilon := rnorm(.N, mean = 0, sd = sqrt(sigma_sq))]

# Define true model parameters
beta_0 <- 2
beta_1 <- 3

# Generate Y values
dt[, Y := beta_0 + beta_1 * X + epsilon]

In [ ]:
# Set background to white
par(bg = "white")  

# Scatter plot to visualize heteroskedasticity
plot(dt$X, dt$Y, main = "Simulated Data with Heteroskedastic Errors", pch = 20)

## Step 2: OLS Estimation
First, estimate the model ignoring heteroskedasticity.

In [ ]:
# Fit OLS model
ols_model <- lm(Y ~ X, data = dt)
summary(ols_model)


In [ ]:
# Set background to white
par(bg = "white")  

# Plot residuals to check heteroskedasticity
plot(dt$X, residuals(ols_model), main = "Residuals vs X (OLS)", pch = 20)

## Step 3: GLS (Weighted Least Squares)

To correct heteroskedasticity, use $1/X^3$ as weights, since:

$$
\sigma^2 \propto X^3
$$


In [ ]:
# Fit GLS model using weighted least squares
gls_model <- lm(Y ~ X, data = dt, weights = 1/dt$X^3)
summary(gls_model)

In [ ]:
# Set background to white
par(bg = "white")  

# Plot residuals to check heteroskedasticity
plot(dt$X, residuals(gls_model)/dt$X^(3/2), main = "Residuals vs X (GLS)", pch = 20)

## Step 4: Alternative weighting


In [ ]:
# Compute weights using inverse squared fitted values
dt[, weights := 1 / fitted(ols_model)^2]

# Fit GLS model with new weights
gls_alt_model <- lm(Y ~ X, data = dt, weights = dt$weights)

# Compare results
summary(gls_alt_model)

# Breusch-Pagan test for new GLS model
#bptest(gls_alt_model)


## Step 5: Compare OLS and GLS
Compare coefficient estimates and standard errors.


In [ ]:
# Create a data.table for comparison
results <- data.table(
  Model = c("OLS", "GLS", "GLS_ALT"),
  Beta_0 = c(coef(ols_model)[1], coef(gls_model)[1], coef(gls_alt_model)[1]),
  Beta_1 = c(coef(ols_model)[2], coef(gls_model)[2], coef(gls_alt_model)[2]),
  Std_Error_Beta_0 = c(summary(ols_model)$coefficients[1, 2], summary(gls_model)$coefficients[1, 2], summary(gls_alt_model)$coefficients[1, 2]),
  Std_Error_Beta_1 = c(summary(ols_model)$coefficients[2, 2], summary(gls_model)$coefficients[2, 2], summary(gls_alt_model)$coefficients[2, 2])
)

print(results)

In [ ]:
# Set background to white
par(bg = "white")  
par(mfrow = c(1,2))
plot(dt$X, residuals(ols_model) , main = "OLS Scaled Residuals", pch = 20, col = "red")
plot(dt$X, residuals(gls_model) / sqrt(dt$X^3), main = "GLS Scaled Residuals", pch = 20, col = "blue")
